In this module, I built a text classification pipeline using a Naive Bayes classifier to predict whether Jeopardy! questions are “high value” or “low value” based on their wording. After loading and cleaning the dataset, I tokenized the text, vectorized it, and trained a baseline model to see how well linguistic features can predict difficulty.

#1) Set up and import.  Loading the environment:

In [4]:
from google.colab import files
uploaded = files.upload()  #uploading from local computer

Saving jeopardy.json to jeopardy.json


In [6]:
import os, random
import numpy as np
import json
import pandas as pd

SEED = 123  # set.seed in R
random.seed(SEED); np.random.seed(SEED)

# Get the uploaded filename (colab sets automatically)
local_name = list(uploaded.keys())[0]

JSON_PATH = f"/content/{local_name}" #construct full path

with open(JSON_PATH, "r", encoding="utf-8") as f: #load json in python
    data = json.load(f)

df = pd.DataFrame(data) #convert json to panas
print("Records:", len(df))
df.head() #sanity checks


Records: 216930


,category,air_date,question,value,answer,round,show_number
0,HISTORY,2004-12-31,"'For the last 8 years of his life, Galileo was...",$200,Copernicus,Jeopardy!,4680
1,ESPN's TOP 10 ALL-TIME ATHLETES,2004-12-31,'No. 2: 1912 Olympian; football star at Carlis...,$200,Jim Thorpe,Jeopardy!,4680
2,EVERYBODY TALKS ABOUT IT...,2004-12-31,'The city of Yuma in this state has a record a...,$200,Arizona,Jeopardy!,4680
3,THE COMPANY LINE,2004-12-31,"'In 1963, live on ""The Art Linkletter Show"", t...",$200,McDonald\'s,Jeopardy!,4680
4,EPITAPHS & TRIBUTES,2004-12-31,"'Signer of the Dec. of Indep., framer of the C...",$200,John Adams,Jeopardy!,4680


2)  Clean value column.  Assign "low" = 0 or "high"  = 1 to value.  $1000 is used as the cutoff, and is the cutoff between first and second round questions.

In [8]:
def clean_value(v):
    # handle missing or non-string values
    if v is None:
        return 0
    if not isinstance(v, str):
        v = str(v)

    v = v.strip().lower()

    # common placeholders in dataset, debugging
    if v in {"", "none", "null"}:
        return 0

    # keep only digits
    digits = re.sub(r"[^\d]", "", v)
    return int(digits) if digits else 0

df["value_clean"] = df["value"].apply(clean_value)
df["high_value"] = (df["value_clean"] >= 1000).astype(int) #assigns 0=low, 1=high values



Check, debugging:

In [9]:
df["value_clean"].max()

18000

In [10]:
print(df["value"].isna().sum(), "rows had value = None")
print(df["value_clean"].describe())

# make sure both classes are present:
df["high_value"].value_counts()


3634 rows had value = None
count    216930.000000
mean        739.988476
std         639.822693
min           0.000000
25%         400.000000
50%         600.000000
75%        1000.000000
max       18000.000000
Name: value_clean, dtype: float64


,count
high_value,
0,155622
1,61308


In [11]:
df[["value", "value_clean", "high_value"]].head()

,value,value_clean,high_value
0,$200,200,0
1,$200,200,0
2,$200,200,0
3,$200,200,0
4,$200,200,0


#3)  Build and test tokenizer:

In [12]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re

# tokenization models (for word_tokenize)
nltk.download("punkt_tab", quiet=True) #gemini debug
nltk.download("stopwords", quiet=True)

STOPWORDS = set(stopwords.words("english")) #set of english stopwords

def simple_tokenizer(text):  #using docstring for function
    """
    Simple tokenizer for Jeopardy questions:
      - lowercase
      - remove punctuation/digits
      - tokenize
      - remove stopwords and 1-letter tokens
    """
    if not isinstance(text, str):
        return []

    text = text.lower() #lowercase
    text = re.sub(r"[^a-z\s]", " ", text) #remove everything except letters/spaces
    tokens = word_tokenize(text) #tokenize
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return tokens


In [ ]:
Sanity check:

In [13]:
print(simple_tokenizer(df.loc[0, "question"]))


['last', 'years', 'life', 'galileo', 'house', 'arrest', 'espousing', 'man', 'theory']


#4) Vectorize the questions

Count Vectorizer transforms text to matrix, where each word is a numeric feature.

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer=simple_tokenizer, min_df=2) #apply cleaned tokenizer, removes rare words
X = vectorizer.fit_transform(df["question"]) #matrix of numeric values
y = df["high_value"] # binary column: low/high value labels


sanity check

In [16]:
print("X shape:", X.shape) #predictors
print("y shape:", y.shape) #outcome

X shape: (216930, 49019)
y shape: (216930,)


In [17]:
df[["question", "high_value"]].head()

,question,high_value
0,"'For the last 8 years of his life, Galileo was...",0
1,'No. 2: 1912 Olympian; football star at Carlis...,0
2,'The city of Yuma in this state has a record a...,0
3,"'In 1963, live on ""The Art Linkletter Show"", t...",0
4,"'Signer of the Dec. of Indep., framer of the C...",0


In [18]:
question = df["question"].iloc[0] #look at tokens first question
tokens = simple_tokenizer(question)
print("Original:", question)
print("Tokens:", tokens)

Original: 'For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory'
Tokens: ['last', 'years', 'life', 'galileo', 'house', 'arrest', 'espousing', 'man', 'theory']


#5) Train/test split and NB Model:

In [46]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

Xtr, Xte, ytr, yte = train_test_split(
    X, y,
    test_size=0.25, #75/25 split
    random_state=SEED, #set seed
    stratify=y  #strata in tidymodels
)

nb = MultinomialNB() #define model, engine...
nb.fit(Xtr, ytr)  #train on training data

preds = nb.predict(Xte)  #produce predicted classes

accuracy = accuracy_score(yte, preds)
print("Naive Bayes accuracy:", round(accuracy, 3)) #requested for assignment


Naive Bayes accuracy: 0.691


The Naive Bayes classifier provides a baseline for this task.  Using a train/test split, the model achieved an accuracy of 0.691, which was expected given the limited cues available for predicting difficulty. Although the performance is modest, the exercise establishes a baseline and sets the stage for more advanced methods in the Module 05 Essentials extension.